In [1]:
import pandas as pd
from sklearn.model_selection import cross_val_predict
from sklearn.ensemble import RandomForestRegressor

## Preparing the entire annotations dataset

In [2]:
df = pd.read_csv('../Datasets/ParlaMint-SI_Annotations.csv', encoding='utf-8')
print(len(df))
df.head()

11765


,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.0,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...","Gospa ministrica, potem se boste lahko javili,...","Madam Minister, you'll be able to come forward...",True,N_Neutral,2.0,2.636969,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Besedo pa ima gospod Gorenak.,Mr. Gorenak has the floor.,True,N_Neutral,2.0,3.639261,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Besedo ima Silva Črnugelj.,"The word is ""silva blackugelj.""",True,P_Neutral,3.0,3.340636,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...


In [3]:
df = df.drop(columns=['text_en', 'chair', 'metadata'])
df.head()
len(df)

11765

In [4]:
aggregation = df.groupby('ID')['annotation_sent'].agg(
    mean = 'mean',
    median = 'median'
)
df = df.merge(aggregation, on='ID')

In [5]:
def count_avg(
    df: pd.DataFrame, id_col: str, sent_col: str, annotation_col: str) -> pd.DataFrame:
    df["count"] = df.groupby(id_col)[sent_col].transform("count")
    count_avg = (
        df.groupby(id_col)
        .apply(lambda x: (x[annotation_col] * x["count"]).sum() / x["count"].sum(), include_groups=False)
        .reset_index(name="count_avg")
    )

    df = df.merge(count_avg, on=id_col, how="left")
    return df


def char_avg(df: pd.DataFrame, id_col: str, sent_col: str, annotation_col: str) -> pd.DataFrame:
    df["char_length"] = df["text_sent"].apply(len)
    char_avg = (
        df.groupby(id_col)
        .apply(lambda x: (x[annotation_col] * x["char_length"]).sum() / x["char_length"].sum(), include_groups=False)
        .reset_index(name='char_avg')
    )
    df = df.merge(char_avg, on=id_col, how='left')
    return df

df = count_avg(df, id_col='ID', sent_col='sent_id', annotation_col='annotation_sent')
df = char_avg(df, id_col='ID', sent_col='sent_id', annotation_col='annotation_sent')
df.head()

,ID,sent_id,text_utt,text_sent,annotation_utt,annotation_utt_score,annotation_sent,mean,median,count,count_avg,char_length,char_avg
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,N_Neutral,2.0,2.240412,2.838881,2.636969,3,2.838881,22,2.803708
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...","Gospa ministrica, potem se boste lahko javili,...",N_Neutral,2.0,2.636969,2.838881,2.636969,3,2.838881,71,2.803708
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Besedo pa ima gospod Gorenak.,N_Neutral,2.0,3.639261,2.838881,2.636969,3,2.838881,29,2.803708
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,P_Neutral,3.0,4.084590,3.491818,3.340636,3,3.491818,11,3.265325
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Besedo ima Silva Črnugelj.,P_Neutral,3.0,3.340636,3.491818,3.340636,3,3.491818,26,3.265325


In [6]:
columns = ['ID', 'annotation_sent', 'annotation_utt_score', 'count', 'char_length', 'mean', 'median', 'char_avg']
df = df[columns]

df.loc[:, 'sum'] = df.groupby('ID')['char_length'].transform('sum')
df.loc[:, 'Q1'] = df.groupby('ID')['annotation_sent'].transform(lambda x: x.quantile(0.25))
df.loc[:, 'Q3'] = df.groupby('ID')['annotation_sent'].transform(lambda x: x.quantile(0.75))
df = df.drop(columns=['char_length'])
df.head()

,ID,annotation_sent,annotation_utt_score,count,mean,median,char_avg,sum,Q1,Q3
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,2.240412,2.0,3,2.838881,2.636969,2.803708,122,2.438690,3.138115
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,2.636969,2.0,3,2.838881,2.636969,2.803708,122,2.438690,3.138115
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,3.639261,2.0,3,2.838881,2.636969,2.803708,122,2.438690,3.138115
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,4.084590,3.0,3,3.491818,3.340636,3.265325,88,3.195431,3.712613
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,3.340636,3.0,3,3.491818,3.340636,3.265325,88,3.195431,3.712613


In [7]:
data = df.drop_duplicates(subset=['ID']).reset_index(drop=True)
print(len(data))
data.head()

1000


,ID,annotation_sent,annotation_utt_score,count,mean,median,char_avg,sum,Q1,Q3
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,2.240412,2.0,3,2.838881,2.636969,2.803708,122,2.438690,3.138115
1,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,4.084590,3.0,3,3.491818,3.340636,3.265325,88,3.195431,3.712613
2,ParlaMint-SI_2011-06-21-SDZ5-Redna-29.ana.u178,4.084590,5.0,15,3.235311,3.166192,3.044246,2442,2.545302,3.812141
3,ParlaMint-SI_2020-09-21-SDZ8-Redna-20.ana.u276,3.971972,0.0,32,1.978221,1.869849,1.788266,3646,0.537923,2.955510
4,ParlaMint-SI_2009-11-18-SDZ5-Redna-11.ana.u120,3.156749,3.0,1,3.156749,3.156749,3.156749,34,3.156749,3.156749


## Retraining the Random Forest model

In [8]:
# Features (count, mean, median, sum, character-based average, Q1, Q2)
X = data[['count', 'sum', 'mean', 'median', 'char_avg', 'Q1', 'Q3']]
y = data['annotation_utt_score']

X_train, X_test = X[:800], X[800:]
y_train, y_test = y[:800], y[800:]

print('Train X:' , len(X_train))
print('Test X:' ,len(X_test))
print('Train y: ',len(y_train))
print('Test y: ', len(y_test))

Train X: 800
Test X: 200
Train y:  800
Test y:  200


In [9]:
rf = RandomForestRegressor(n_estimators=100)
y_pred = cross_val_predict(rf, X, y, cv=5)

## Fitting the model

In [13]:
rf.fit(X_train, y_train)
test_predictions = rf.predict(X_test)

In [17]:
import joblib

joblib.dump(rf, '../Models/RF_retrained.pkl')

['../Models/RF_retrained.pkl']